# Evaluation: Latent Domain Recovery

Load a trained model and analyse:
- The learned lifting map $L$
- The group generators $G$
- The lifted representation on test data
- Alignment of the learned structure with ground-truth group action

## 1. Setup

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from core.train_utils import create_model, make_data_generator

print('TensorFlow:', tf.__version__)

## 2. Select experiment to evaluate

Point to a directory under `experiments/` that contains `specs.json` and a saved weights file.

In [ ]:
EXP_NAME = 'AFTER-MERGE-MATRIX-GAUSSIAN-IDENTITY'  # change to your experiment
EPOCH    = None   # None -> pick the latest available checkpoint

exp_dir    = os.path.join(REPO_ROOT, 'experiments', EXP_NAME)
epochs_dir = os.path.join(exp_dir, 'epochs')
specs_path = os.path.join(exp_dir, 'specs.json')

with open(specs_path) as f:
    exp_specs = json.load(f)

# Find available checkpoints
h5_files = sorted([f for f in os.listdir(epochs_dir) if f.endswith('.h5')])
print(f'Available checkpoints: {h5_files}')

if EPOCH is None:
    weights_path = os.path.join(epochs_dir, h5_files[-1])
else:
    weights_path = os.path.join(epochs_dir, f'ep{EPOCH}.h5')

print(f'Loading weights from: {weights_path}')

## 3. Rebuild model and load weights

In [ ]:
seed = exp_specs.get('seed', 0)
np.random.seed(seed)
tf.random.set_seed(seed)

model = create_model(**exp_specs['model_params'])

# Build model by running a forward pass
dg_tmp = make_data_generator(seed=seed, **exp_specs['data_generator_params'])
sample = tf.constant(dg_tmp.sample_batch_of_data(), dtype=tf.float32)
_ = model(sample, training=False)

model.load_weights(weights_path)
print('Weights loaded.')
model.summary()

## 4. Generate test data

In [ ]:
dg = make_data_generator(seed=seed + 999, **exp_specs['data_generator_params'])
dg.reset_batch_counter()
x_test = np.concatenate([dg.sample_batch_of_data() for _ in range(5)], axis=0)
print(f'Test data shape: {x_test.shape}')

## 5. Lifting map

In [ ]:
x_tf = tf.constant(x_test[:64], dtype=tf.float32)
lifted, L = model(x_tf, training=False, analyze=True)

L_np = L.numpy()   # shape: (input_dim, group_positions * channels)

print(f'Lifting map L shape: {L_np.shape}')
print(f'Lifted output shape: {lifted.shape}')

plt.figure(figsize=(9, 3))
plt.imshow(L_np.T, aspect='auto', cmap='RdBu_r')
plt.colorbar()
plt.xlabel('Input dimension')
plt.ylabel('Group position')
plt.title('Learned lifting map $L$')
plt.tight_layout(); plt.show()

## 6. Generator matrices

In [ ]:
G     = model.lifting_layer.generators.numpy()      # (group_dims, D, D)
log_G = model.lifting_layer.log_generators.numpy()  # (group_dims, D, D)
phases = model.lifting_layer.generator_phases.numpy()  # (group_dims, D)

fig, axes = plt.subplots(G.shape[0], 3, figsize=(14, 4 * G.shape[0]))
if G.shape[0] == 1:
    axes = [axes]

for d, row in enumerate(axes):
    im0 = row[0].imshow(G[d], cmap='RdBu_r', vmin=-1, vmax=1)
    row[0].set_title(f'Generator $G_{d}$')
    plt.colorbar(im0, ax=row[0])

    im1 = row[1].imshow(log_G[d], cmap='RdBu_r')
    row[1].set_title(f'Log-generator $\\log G_{d}$')
    plt.colorbar(im1, ax=row[1])

    row[2].plot(np.sort(phases[d]))
    row[2].set_title(f'Generator eigenvalue phases (dim {d})')
    row[2].set_xlabel('Eigenvalue index'); row[2].set_ylabel('Phase (rad)')
    row[2].grid(True)

plt.suptitle('Learned group generators', y=1.01)
plt.tight_layout(); plt.show()

## 7. Lifted representation samples

In [ ]:
lifted_np = lifted.numpy()   # (batch, positions, channels)
n_show = min(8, lifted_np.shape[0])

fig, axes = plt.subplots(2, n_show // 2, figsize=(14, 5))
axes = axes.ravel()

for i in range(n_show):
    axes[i].plot(lifted_np[i, :, 0])
    axes[i].set_title(f'Sample {i}')
    axes[i].set_xlabel('Group position')

plt.suptitle('Lifted representation (channel 0) for test samples')
plt.tight_layout(); plt.show()

## 8. Generator alignment metric

A well-learned 1-D translation group has $G$ whose rows are cyclic shifts of one another  
(i.e. $G$ is circulant). We measure this by checking whether each row of $G$ maximally  
correlates with a shifted version of the first row.

In [ ]:
if G.shape[0] == 1:
    g = G[0]  # (D, D)
    D = g.shape[0]

    # Cross-correlate each row of G with row 0
    ref = g[0] - g[0].mean()
    corrs = []
    for i in range(D):
        row = g[i] - g[i].mean()
        xc = np.correlate(ref, row, mode='full')
        corrs.append(np.max(np.abs(xc)) / (np.linalg.norm(ref) * np.linalg.norm(row) + 1e-8))

    corrs = np.array(corrs)
    print(f'Mean peak cross-correlation between rows of G: {corrs.mean():.4f}  (1.0 = perfect circulant)')

    plt.figure(figsize=(6, 3))
    plt.plot(corrs, '-o', markersize=4)
    plt.axhline(1.0, ls='--', color='gray', label='Perfect circulant')
    plt.xlabel('Row index'); plt.ylabel('Peak cross-correlation')
    plt.title('Row-wise circulant alignment of $G$')
    plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()
else:
    print('Circulant alignment metric shown only for 1-D group (group_dims=1).')